# Experiment V2: YOLOv8 Deeper Backbone for Object Detection

**Student ID:** 25509225  
**Experiment:** V2 - YOLOv8 with Deeper Backbone  
**Model:** YOLOv8m + 2 additional convolutional layers

---

## Overview

This experiment trains a YOLOv8m model with a deeper backbone architecture. Two extra convolutional layers are added after backbone layer 2 to enhance shallow-layer feature extraction. This is compared with the baseline (V1) and shallower backbone (V3) models.

## Cell 1: Load Modules

In [ ]:
# Load shared modules from YOLOv8_modules.ipynb
%run ./YOLOv8_modules.ipynb

print("✓ All modules loaded successfully")

## Cell 2: Configuration

In [ ]:
# ============================================================================
# Experiment V2: YOLOv8 Deeper Backbone Configuration
# ============================================================================

# === Model Configuration ===
YOLOV8_V2_MODEL_CONFIG = {
    'backbone': 'm',
    'input_size': 640,
    'confidence_threshold': 0.5,
    'nms_iou_threshold': 0.45,
    'pretrained': True,
    'customize_type': 'deeper'  # Adds 2 conv layers to backbone
}

# === Training Configuration ===
TRAINING_CONFIG_V2 = {
    'learning_rate': 0.0005,     # Lower LR for deeper model stability
    'batch_size': 8,             # Smaller batch due to larger model (T4 GPU)
    'epochs': 300,
    'optimizer': 'adam',
    'weight_decay': 5e-4,        # Higher weight decay to prevent overfitting
    'use_amp': True,             # Mixed precision
    'patience': 50,              # Early stopping patience
    'cos_lr': True,              # Cosine LR schedule for better convergence
    'close_mosaic': 10           # Close mosaic in last 10 epochs
}

# === Experiment Settings ===
STUDENT_ID = "25509225"
DATASET_CONFIG = f"/home/sagemaker-user/CNN_A2/data/{STUDENT_ID}/Object_Detection/yolo/data.yaml"
USE_PRETRAINED = True

# Output directory (simplified, no timestamp subdirectory)
output_dir = Path('/home/sagemaker-user/CNN_A2/notebooks/detection_YOLOv8/outputs/detection_yolov8_v2')
output_dir.mkdir(parents=True, exist_ok=True)

# Print experiment info
print("=" * 80)
print("EXPERIMENT V2: YOLOv8 Deeper Backbone")
print("=" * 80)
print(f"\nExperiment Settings:")
print(f"  Student ID: {STUDENT_ID}")
print(f"  Dataset Config: {DATASET_CONFIG}")
print(f"  Output Directory: {output_dir}")
print(f"\nModel Configuration:")
print(f"  Backbone: YOLOv8{YOLOV8_V2_MODEL_CONFIG['backbone']} + Deeper")
print(f"  Input Size: {YOLOV8_V2_MODEL_CONFIG['input_size']}")
print(f"  Pretrained: {USE_PRETRAINED}")
print(f"  Customization: Deeper (2 extra conv layers)")
print(f"\nTraining Configuration:")
print(f"  Learning Rate: {TRAINING_CONFIG_V2['learning_rate']}")
print(f"  Batch Size: {TRAINING_CONFIG_V2['batch_size']}")
print(f"  Epochs: {TRAINING_CONFIG_V2['epochs']}")
print(f"  Optimizer: {TRAINING_CONFIG_V2['optimizer']}")
print(f"  Weight Decay: {TRAINING_CONFIG_V2['weight_decay']}")
print(f"  Mixed Precision: {TRAINING_CONFIG_V2['use_amp']}")
print(f"  Early Stopping: {TRAINING_CONFIG_V2['patience']} epochs")
print(f"  Cosine LR: {TRAINING_CONFIG_V2['cos_lr']}")
print(f"  Close Mosaic: {TRAINING_CONFIG_V2['close_mosaic']} epochs")

## Cell 3: Step 1 - Load Dataset Configuration

In [ ]:
# ============================================================================
# Step 1: Load Dataset Configuration
# ============================================================================

print("\n[1/5] Loading dataset configuration...")
print("=" * 80)

dataset_config_path = Path(DATASET_CONFIG)

if not dataset_config_path.exists():
    print(f"Error: Dataset config not found: {dataset_config_path}")
    print("Please check the DATASET_CONFIG path in Cell 2")
    raise FileNotFoundError(f"Dataset config not found: {dataset_config_path}")

with open(dataset_config_path, 'r') as f:
    dataset_config = yaml.safe_load(f)

print(f"Dataset config: {dataset_config_path}")
print(f"Number of classes: {dataset_config['nc']}")
print(f"Class names: {dataset_config['names']}")
print(f"\nPaths:")
print(f"  Train: {dataset_config.get('train', 'N/A')}")
print(f"  Val: {dataset_config.get('val', 'N/A')}")
print(f"  Test: {dataset_config.get('test', 'N/A')}")
print("\n✓ Dataset configuration loaded successfully")

## Cell 4: Step 2 - Initialize Model

In [ ]:
# ============================================================================
# Step 2: Initialize YOLOv8 Model with Deeper Backbone
# ============================================================================

print("\n[2/5] Initializing YOLOv8 model with deeper backbone...")
print("=" * 80)

# Update model config with the pretrained setting
model_config = YOLOV8_V2_MODEL_CONFIG.copy()
model_config['pretrained'] = USE_PRETRAINED

# Create model
model = YOLOv8Detector(**model_config)

print(f"\nModel Information:")
print(f"  Model: YOLOv8{model_config['backbone']} + Deeper Backbone")
print(f"  Input Size: {model_config['input_size']}")
print(f"  Pretrained: {USE_PRETRAINED}")
print(f"  Customization: Deeper (2 extra conv layers after layer 2)")
print(f"  Confidence Threshold: {model_config['confidence_threshold']}")
print(f"  NMS IoU Threshold: {model_config['nms_iou_threshold']}")

# Calculate approximate parameters
total_params = sum(p.numel() for p in model.model.parameters())
print(f"\n  Total Parameters: ~{total_params/1e6:.1f}M")
print("\n✓ Model initialized successfully")

## Cell 5: Step 3 - Train Model

In [ ]:
# ============================================================================
# Step 3: Train Model
# ============================================================================

print("\n[3/5] Training model...")
print("=" * 80)

# Initialize trainer
trainer = YOLOv8Trainer(**TRAINING_CONFIG_V2)

# Train model
training_results = trainer.train(
    model=model,
    train_data=DATASET_CONFIG,
    val_data=DATASET_CONFIG,
    output_dir=str(output_dir)
)

# Print training summary
print("\n" + "=" * 80)
print("TRAINING COMPLETED")
print("=" * 80)
print(f"\nTraining Results:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")
print(f"  Output Directory: {output_dir}")
print("\n✓ Model training completed successfully")

## Cell 6: Step 4 - Evaluate on Test Set

In [ ]:
# ============================================================================
# Step 4: Evaluate on Test Set
# ============================================================================

print("\n[4/5] Evaluating on test set...")
print("=" * 80)

# Initialize evaluator
evaluator = DetectionEvaluator()

# Evaluate model
metrics = evaluator.evaluate_yolov8(
    model=model,
    test_dataset=DATASET_CONFIG
)

print("\n" + "=" * 80)
print("EVALUATION COMPLETED")
print("=" * 80)
print(f"\nTest Set Metrics:")
print(f"  mAP@0.5: {metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"  F1-Score: {metrics['f1']:.4f}")
print("\n✓ Model evaluation completed successfully")

## Cell 7: Display Detection Results

In [ ]:
# ============================================================================
# Visualization 1: Detection Results with Bounding Boxes
# ============================================================================

print("\n=== Detection Results Visualization ===")
print("=" * 80)

# Plot detection results
fig = evaluator.plot_detection_results(model, DATASET_CONFIG, num_samples=5)
if fig is not None:
    plt.show()
    print("\n✓ Detection visualization displayed")
else:
    print("Warning: Could not generate detection visualization")

## Cell 8: Display Training Loss Curves

In [ ]:
# ============================================================================
# Visualization 2: Training Loss Curves (Box Loss + Classification Loss)
# ============================================================================

print("\n=== Training Loss Curves ===")
print("=" * 80)

# Store training output path for visualization
train_output_dir = output_dir / 'training'

# Plot training loss curves
fig = evaluator.plot_training_losses(str(train_output_dir))
if fig is not None:
    plt.show()
    print("\n✓ Training loss curves displayed")
else:
    print("Warning: Could not generate training loss curves")
    print("This may be because training results.csv is not available yet.")
    print("Try running the training cell (Cell 5) first if you haven't already.")

## Cell 9: Display Validation mAP Curves

In [ ]:
# ============================================================================
# Visualization 3: Validation mAP Curves (mAP@0.5 + mAP@0.5:0.95)
# ============================================================================

print("\n=== Validation mAP Curves ===")
print("=" * 80)

# Plot validation mAP curves
fig = evaluator.plot_validation_mAP(str(train_output_dir))
if fig is not None:
    plt.show()
    print("\n✓ Validation mAP curves displayed")
else:
    print("Warning: Could not generate validation mAP curves")
    print("This may be because training results.csv is not available yet.")
    print("Try running the training cell (Cell 5) first if you haven't already.")

## Cell 9.5: Display Confusion Matrix

In [ ]:
# ============================================================================
# Visualization 4: Confusion Matrix (5-class Detection)
# ============================================================================

print("\n=== Confusion Matrix ===")
print("=" * 80)

# Plot confusion matrix
fig = evaluator.plot_confusion_matrix(model, DATASET_CONFIG)
if fig is not None:
    plt.show()
    print("\n✓ Confusion matrix displayed")
else:
    print("Warning: Could not generate confusion matrix")
    print("This may be because the model evaluation did not complete successfully.")

In [ ]:
# ============================================================================
# Visualization 5: Precision-Recall Curves
# ============================================================================

print("\n=== Precision-Recall Curves ===")
print("=" * 80)

# Plot PR curves from training results
fig = evaluator.plot_pr_curves_from_results(str(train_output_dir))
if fig is not None:
    plt.show()
    print("\n✓ PR curves displayed")
else:
    print("Warning: Could not generate PR curves")
    print("This may be because PR_curve.png was not generated during training.")

In [ ]:
# ============================================================================
# Analysis: Model Characteristics and Performance
# ============================================================================

print("\n" + "=" * 80)
print("MODEL ANALYSIS")
print("=" * 80)

# Model characteristics
total_params = sum(p.numel() for p in model.model.parameters())

print(f"\nModel Characteristics:")
print(f"  Backbone: YOLOv8{model_config['backbone']} + Deeper")
print(f"  Input Size: {model_config['input_size']}")
print(f"  Total Parameters: ~{total_params/1e6:.1f}M")
print(f"  Customization: Deeper Backbone (2 extra conv layers)")
print(f"  Pretrained Weights: {USE_PRETRAINED}")

print(f"\nPerformance Summary:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")
print(f"  Test mAP@0.5: {metrics['map50']:.4f}")
print(f"  Test mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Test Precision: {metrics['precision']:.4f}")
print(f"  Test Recall: {metrics['recall']:.4f}")
print(f"  Test F1-Score: {metrics['f1']:.4f}")

print("\n" + "=" * 80)
print("DEEPER BACKBONE ANALYSIS")
print("=" * 80)
print("\nThis model has a deeper backbone with 2 extra convolutional layers")
print("added after backbone layer 2. This should enhance shallow-layer")
print("feature extraction and potentially improve detection accuracy.")
print("\nCompared with:")
print("  - V1: YOLOv8m Baseline (no modifications)")
print("  - V3: YOLOv8m with Shallower Backbone (reduced conv layers)")
print("\n✓ Analysis completed")

## Cell 11: Final Summary

In [ ]:
# ============================================================================
# Final Summary
# ============================================================================

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED")
print("=" * 80)

print(f"\nExperiment: V2 - YOLOv8 Deeper Backbone")
print(f"Model: YOLOv8{model_config['backbone']} + Deeper")
print(f"Pretrained: {USE_PRETRAINED}")
print(f"Customization: Deeper Backbone (2 extra conv layers)")

print(f"\n" + "-" * 80)
print(f"Training Performance:")
print(f"  Best mAP@0.5: {training_results.box.map50:.4f}")
print(f"  Best mAP@0.5:0.95: {training_results.box.map:.4f}")

print(f"\n" + "-" * 80)
print(f"Test Set Performance:")
print(f"  mAP@0.5: {metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {metrics['map50_95']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall: {metrics['recall']:.4f}")
print(f"  F1-Score: {metrics['f1']:.4f}")

print(f"\n" + "-" * 80)
print(f"Output Directory: {output_dir}")
print("\n" + "=" * 80)
print("✓ All experiments completed successfully")
print("=" * 80)